<a href="https://colab.research.google.com/github/shiv1122004/ML/blob/main/day32-binning-and-binarization/binarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import numpy as np
import pandas as pd

In [25]:
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score

from sklearn.compose import ColumnTransformer

In [27]:
# https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day32-binning-and-binarization/train.csv
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/100-days-of-machine-learning/refs/heads/main/day32-binning-and-binarization/train.csv')[['Age','Fare','SibSp','Parch','Survived']]

In [28]:
df.dropna(inplace=True)

In [29]:
df.head()

,Age,Fare,SibSp,Parch,Survived
0,22.0,7.2500,1,0,0
1,38.0,71.2833,1,0,1
2,26.0,7.9250,0,0,1
3,35.0,53.1000,1,0,1
4,35.0,8.0500,0,0,0


In [30]:
df['family'] = df['SibSp'] + df['Parch']

In [31]:
df.head()

,Age,Fare,SibSp,Parch,Survived,family
0,22.0,7.2500,1,0,0,1
1,38.0,71.2833,1,0,1,1
2,26.0,7.9250,0,0,1,0
3,35.0,53.1000,1,0,1,1
4,35.0,8.0500,0,0,0,0


In [32]:
df.drop(columns=['SibSp','Parch'],inplace=True)

In [33]:
df.head()

,Age,Fare,Survived,family
0,22.0,7.2500,0,1
1,38.0,71.2833,1,1
2,26.0,7.9250,1,0
3,35.0,53.1000,1,1
4,35.0,8.0500,0,0


In [37]:
X = df.drop(columns=['Survived'])
y = df['Survived']

In [38]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [39]:
X_train.head()

,Age,Fare,family
328,31.0,20.5250,2
73,26.0,14.4542,1
253,30.0,16.1000,1
719,33.0,7.7750,0
666,25.0,13.0000,0


In [40]:
# Without binarization

clf = DecisionTreeClassifier()

clf.fit(X_train,y_train)

y_pred = clf.predict(X_test)

accuracy_score(y_test,y_pred)

0.6503496503496503

In [41]:
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6443075117370893)

In [42]:
# Applying Binarization

from sklearn.preprocessing import Binarizer

In [58]:
trf = ColumnTransformer([                         #this will ensure if the person is travelling alone or with family
    ('bin',Binarizer(copy=False),['family'])
],remainder='passthrough')

In [59]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [60]:
pd.DataFrame(X_train_trf,columns=['family','Age','Fare'])
df.sample(10)

,Age,Fare,Survived,family
763,36.0,120.0000,1,3
438,64.0,263.0000,0,5
698,49.0,110.8833,0,2
861,21.0,11.5000,0,1
855,18.0,9.3500,1,1
820,52.0,93.5000,1,2
752,33.0,9.5000,0,0
243,22.0,7.1250,0,0
804,27.0,6.9750,1,0
805,31.0,7.7750,0,0


In [62]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

accuracy_score(y_test,y_pred2)

0.6013986013986014

In [63]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X_trf,y,cv=10,scoring='accuracy'))

np.float64(0.6317879499217527)